# ChronoState-KD Lite — Phase 1: Teacher logit cache + teacher eval

**Goal:** build the two training sets (code, reasoning), run the 3B teacher over them **once**, store its top-64 logits per token on Drive, and record the teacher's own scores on the eval sets. After this notebook the teacher is never loaded again.

**Budget:** ~40 min on a T4 (≈10 min caching, ≈25 min teacher eval, rest is downloads).

**Gate to pass before Phase 2:**
1. `data/code.pt` and `data/reasoning.pt` on Drive, ~1,500 rows each
2. All shards present for both domains; `verify_shard` top-1 agreement well above 50 %
3. `results/teacher_eval.json` written (GSM8K-100 accuracy, MBPP-50 pass@1)

Runtime → **T4 GPU** first. Edit `GITHUB_URL`.

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_GITHUB_USER>/chronostate-kd.git"   # <-- edit
REPO_DIR   = "/content/chronostate-kd"
DRIVE_ROOT = "/content/drive/MyDrive/chronostate"
CONFIG     = "configs/lite.yaml"


## 1. Setup (same as Phase 0, condensed)

In [ ]:
import os, torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
if not os.path.exists(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt && pip install -q -e .
from google.colab import drive
drive.mount("/content/drive")
for sub in ["data", "teacher_cache", "adapters", "ckpt", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)


In [ ]:
import json, time, torch
from chronostate.config import load_config, to_dict
from chronostate.monitor import Monitor
cfg = load_config(CONFIG)
mon = Monitor(f"{DRIVE_ROOT}/results/events.jsonl", run="phase1-teacher")
print(json.dumps(to_dict(cfg)["models"], indent=2))


## 2. Build and save the datasets
Tokenised once with the (shared) tokenizer, prompt tokens masked to −100, saved to Drive. Phase 2 **loads these files** instead of re-tokenising, so row *i* of the teacher cache is guaranteed to match row *i* of the training batch.

In [ ]:
from transformers import AutoTokenizer
from chronostate.data import build_dataset, dataset_stats, save_dataset, load_saved_dataset
tok = AutoTokenizer.from_pretrained(cfg.models.student)      # identical to the teacher's (asserted in Phase 0)
datasets = {}
for domain in ["code", "reasoning"]:
    path = f"{DRIVE_ROOT}/data/{domain}.pt"
    if os.path.exists(path):
        datasets[domain] = load_saved_dataset(path); print(f"[{domain}] loaded existing {path}")
    else:
        t0 = time.time()
        datasets[domain] = build_dataset(cfg, tok, domain)
        save_dataset(datasets[domain], path)
        print(f"[{domain}] built + saved in {time.time()-t0:.0f}s")
    st = dataset_stats(datasets[domain]); print(domain, st)
    mon.log("dataset", domain=domain, **st)


Eyeball one example per domain: the masked prefix must end exactly where the assistant turn begins.

In [ ]:
for domain in ["code", "reasoning"]:
    ex = datasets[domain][0]
    n = ex["n_prompt"]
    print(f"=== {domain}: prompt tokens {n}, total {len(ex['input_ids'])}")
    print("PROMPT ...", repr(tok.decode(ex["input_ids"][max(0,n-30):n])))
    print("RESPONSE  ", repr(tok.decode([t for t in ex["labels"][n:n+40] if t >= 0])))


## 3. Load the teacher (NF4, fp16 compute) and assert the tokenizer once more

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
t_tok = AutoTokenizer.from_pretrained(cfg.models.teacher)
assert t_tok.get_vocab() == tok.get_vocab(), "teacher/student vocab mismatch - STOP"
t0 = time.time()
teacher = AutoModelForCausalLM.from_pretrained(cfg.models.teacher, quantization_config=bnb,
                                               attn_implementation=cfg.models.attn_implementation,
                                               device_map={"": 0})
teacher.eval()
print(f"teacher loaded in {time.time()-t0:.0f}s, VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. Cache top-k logits (resumable by shard)
If the session dies, just re-run this cell: existing shards are skipped.

In [ ]:
from chronostate.teacher import cache_teacher_logits, verify_shard, cache_summary
torch.cuda.reset_peak_memory_stats()
for domain in ["code", "reasoning"]:
    paths = cache_teacher_logits(teacher, datasets[domain], f"{DRIVE_ROOT}/teacher_cache", domain,
                                 pad_id=tok.pad_token_id, max_len=cfg.data.max_len, top_k=cfg.kd.top_k,
                                 batch_size=cfg.teacher_cache.batch_size, shard_rows=cfg.teacher_cache.shard_rows,
                                 monitor=mon)
    rep = verify_shard(paths[0], datasets[domain])
    print(domain, "verify:", rep)
    mon.log("teacher_verify", domain=domain, **rep)
    assert rep["teacher_top1_agreement"] > 0.5, "teacher cache looks misaligned - STOP"
    print(domain, cache_summary(f"{DRIVE_ROOT}/teacher_cache", domain))
print(f"peak VRAM during caching: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


## 5. Teacher baseline on the eval sets
These are the 'Teacher' column of the results table. Batched greedy decoding, left-padded.

In [ ]:
from chronostate.evaluate import eval_gsm8k, eval_mbpp, make_hf_generate
gen = make_hf_generate(teacher, t_tok, max_new_tokens=320)
res = {"model": cfg.models.teacher, "variant": "teacher"}
r = eval_gsm8k(gen, n=cfg.eval.gsm8k_n, batch_size=8, out_jsonl=f"{DRIVE_ROOT}/results/eval_teacher_gsm8k.jsonl", tag="teacher")
print(r); res["gsm8k"] = r; mon.log("eval", variant="teacher", **r)
r = eval_mbpp(gen, n=cfg.eval.mbpp_n, batch_size=8, out_jsonl=f"{DRIVE_ROOT}/results/eval_teacher_mbpp.jsonl", tag="teacher")
print(r); res["mbpp"] = r; mon.log("eval", variant="teacher", **r)
os.makedirs("results", exist_ok=True)
for p in ["results/teacher_eval.json", f"{DRIVE_ROOT}/results/teacher_eval.json"]:
    json.dump(res, open(p, "w"), indent=2)
print(json.dumps(res, indent=2))


## 6. Free the teacher, snapshot the cache manifest

In [ ]:
del teacher, gen; torch.cuda.empty_cache()
summary = {d: cache_summary(f"{DRIVE_ROOT}/teacher_cache", d) for d in ["code", "reasoning"]}
summary["datasets"] = {d: dataset_stats(datasets[d]) for d in datasets}
json.dump(summary, open("results/teacher_cache_summary.json", "w"), indent=2)
json.dump(summary, open(f"{DRIVE_ROOT}/results/teacher_cache_summary.json", "w"), indent=2)
mon.close()
print(json.dumps(summary, indent=2))
!ls -la $DRIVE_ROOT/teacher_cache/* $DRIVE_ROOT/data


## Gate checklist
- [ ] `data/code.pt`, `data/reasoning.pt` on Drive (~1,500 rows each; a few hundred fewer for code is fine if long prompts were skipped)
- [ ] 3 shards per domain (500 rows each) in `teacher_cache/<domain>/`
- [ ] `teacher_top1_agreement` > 0.5 for both domains
- [ ] `results/teacher_eval.json` + `results/teacher_cache_summary.json` downloaded and committed

Report back: the two `verify:` lines, the two eval dicts, peak VRAM, and total wall time. Then Phase 2.